# NeMo Data Designer – Synthetic Data Generation Example

This notebook demonstrates how to use **NeMo Data Designer** (NDD) stage to generate synthetic medical-notes data from a small seed dataset.

The pipeline:
1. Downloads a public symptom-to-diagnosis CSV and converts it to JSONL shards.
2. Define the configuration for the NDD stage.
3. Build and run a Curator pipeline, composing of three stages: `JsonlReader`, `DataDesignerStage`, `JsonlWriter`.

### Imports libraries

In [1]:
import time
from pathlib import Path

import data_designer.config as dd
import pandas as pd

from nemo_curator.core.client import RayClient
from nemo_curator.pipeline import Pipeline
from nemo_curator.stages.synthetic.nemo_data_designer.data_designer import DataDesignerStage
from nemo_curator.stages.text.io.reader.jsonl import JsonlReader
from nemo_curator.stages.text.io.writer.jsonl import JsonlWriter
from nemo_curator.utils.file_utils import get_all_file_paths_under

/tmp/huvu-venvs/curator_env/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
/tmp/huvu-venvs/curator_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-17 09:48:08,488	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


### Download and Prepare Seed Data

The seed dataset is a public symptom-to-diagnosis CSV hosted on GitHub.  
We download it, split it into small JSONL shards (10 rows each), and save them locally so the pipeline can read them in parallel.
We sample 200 records from the original dataset for testing purposes.

In [2]:
SEED_CSV_URL = (
    "https://raw.githubusercontent.com/NVIDIA/GenerativeAIExamples/refs/heads/main"
    "/nemo/NeMo-Data-Designer/data/gretelai_symptom_to_diagnosis.csv"
)


def download_and_convert_seed_data(
    output_dir: str | Path | None = None,
    records_per_file: int = 10,
    number_of_records: int = 200,
) -> str:
    """Download seed CSV from URL, convert to JSONL (chunked), return output dir path."""
    if output_dir is None:
        output_dir = Path("processed_seed_data")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(SEED_CSV_URL, sep=",", encoding="utf-8")
    df = df.head(number_of_records)
    for i, start in enumerate(range(0, len(df), records_per_file)):
        chunk = df.iloc[start : start + records_per_file]
        chunk.to_json(
            output_dir / f"{i:06d}.jsonl",
            orient="records",
            lines=True,
            force_ascii=False,
            date_format="iso",
        )
    return str(output_dir)


print("Preparing seed data (download + CSV→JSONL)...")
seed_dir = download_and_convert_seed_data()
print(f"Seed data ready: {seed_dir}")

Preparing seed data (download + CSV→JSONL)...
Seed data ready: processed_seed_data


### Configuration

Set the output directory for generated data here.  
You can also point `DATA_DESIGNER_CONFIG_FILE` at a YAML config file; leave it as `None` to use the programmatic config defined in this notebook.

In [3]:
OUTPUT_PATH = "./synthetic_output"
DATA_DESIGNER_CONFIG_FILE = None  # set to a file path string to load a YAML config

### Define the Data Designer Config

The DataDesignerStage is intialized from a NDD's configuration builder (https://nvidia-nemo.github.io/DataDesigner/latest/code_reference/config_builder/). In this step, we manually create a configuration for the NDD module. This includes setting the LLM model provider, and the steps taken inside NDD module. Users can learn more about NDD's API from NDD official documentation: https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/.

In this tutorial, we replicate the official NDD's tutorials on synthetic data genertion with seed data (https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/3-seeding-with-a-dataset/), but with Nemo-Curator.


In [4]:
def build_config_manually() -> dd.DataDesignerConfigBuilder:
    """Build the Data Designer config programmatically for medical-notes generation."""
    model_provider = "nvidia"
    model_id = "meta/llama-3.3-70b-instruct"
    model_alias = "llama-3.3-70b"

    model_configs = [
        dd.ModelConfig(
            alias=model_alias,
            model=model_id,
            provider=model_provider,
            inference_parameters=dd.ChatCompletionInferenceParams(
                temperature=1.0,
                top_p=1.0,
                max_tokens=2048,
            ),
        )
    ]

    config_builder = dd.DataDesignerConfigBuilder(model_configs=model_configs)

    # --- Sampler columns (faker / random) ---
    config_builder.add_column(
        dd.SamplerColumnConfig(
            name="patient_sampler",
            sampler_type=dd.SamplerType.PERSON_FROM_FAKER,
            params=dd.PersonFromFakerSamplerParams(),
        )
    )
    config_builder.add_column(
        dd.SamplerColumnConfig(
            name="doctor_sampler",
            sampler_type=dd.SamplerType.PERSON_FROM_FAKER,
            params=dd.PersonFromFakerSamplerParams(),
        )
    )
    config_builder.add_column(
        dd.SamplerColumnConfig(
            name="patient_id",
            sampler_type=dd.SamplerType.UUID,
            params=dd.UUIDSamplerParams(prefix="PT-", short_form=True, uppercase=True),
        )
    )

    # --- Expression columns (Jinja2 references) ---
    config_builder.add_column(dd.ExpressionColumnConfig(name="first_name", expr="{{ patient_sampler.first_name}}"))
    config_builder.add_column(dd.ExpressionColumnConfig(name="last_name",  expr="{{ patient_sampler.last_name }}"))
    config_builder.add_column(dd.ExpressionColumnConfig(name="dob",        expr="{{ patient_sampler.birth_date }}"))

    # --- Date / timedelta samplers ---
    config_builder.add_column(
        dd.SamplerColumnConfig(
            name="symptom_onset_date",
            sampler_type=dd.SamplerType.DATETIME,
            params=dd.DatetimeSamplerParams(start="2024-01-01", end="2024-12-31"),
        )
    )
    config_builder.add_column(
        dd.SamplerColumnConfig(
            name="date_of_visit",
            sampler_type=dd.SamplerType.TIMEDELTA,
            params=dd.TimeDeltaSamplerParams(
                dt_min=1,
                dt_max=30,
                reference_column_name="symptom_onset_date",
            ),
        )
    )
    config_builder.add_column(
        dd.ExpressionColumnConfig(name="physician", expr="Dr. {{ doctor_sampler.last_name }}")
    )

    # --- LLM-generated column ---
    config_builder.add_column(
        dd.LLMTextColumnConfig(
            name="physician_notes",
            prompt="""\
You are a primary-care physician who just had an appointment with {{ first_name }} {{ last_name }},
who has been struggling with symptoms from {{ diagnosis }} since {{ symptom_onset_date }}.
The date of today's visit is {{ date_of_visit }}.

{{ patient_summary }}

Write careful notes about your visit with {{ first_name }},
as Dr. {{ doctor_sampler.first_name }} {{ doctor_sampler.last_name }}.

Format the notes as a busy doctor might.
Respond with only the notes, no other text.
""",
            model_alias=model_alias,
        )
    )

    return config_builder

### Build the NeMo Curator Pipeline

After defining the NDD configuration which will be used to initialize `DataDesignerStage` stage, we build the pipeline chains three stages:

| # | Stage | Role |
|---|-------|------|
| 1 | `JsonlReader` | Read seed JSONL shards from disk |
| 2 | `DataDesignerStage` | Enrich each record using the NDD config |
| 3 | `JsonlWriter` | Write enriched records back to JSONL |

Note: This notebook utilizes a free, publicly available NVIDIA LLM endpoint. To prevent exceeding rate limits, the number of concurrent actors in the DataDesignerStage is intentionally restricted (through setting `xenna_stage_spec`). When using a commercial-grade endpoint, these constraints can be relaxed or removed to achieve optimal performance.

In [ ]:
pipeline = Pipeline(
    name="ndd_data_generation",
    description="Generate synthetic text data using Nemo Data Designer",
)

pipeline.add_stage(
    JsonlReader(
        file_paths=seed_dir + "/*.jsonl",
        fields=["diagnosis", "patient_summary"],
    )
)

if DATA_DESIGNER_CONFIG_FILE is not None:
    config_builder = dd.DataDesignerConfigBuilder.from_config(DATA_DESIGNER_CONFIG_FILE)
else:
    config_builder = build_config_manually()

# Add the Nemo Data Designer stage (limit to 1 actor to stay within API rate limits)
ndd_stage = DataDesignerStage(config_builder=config_builder)
ndd_stage.xenna_stage_spec = lambda: {"num_workers": 3}
pipeline.add_stage(ndd_stage)

pipeline.add_stage(JsonlWriter(path=OUTPUT_PATH))

print(pipeline.describe())

2026-03-17 09:48:18.774 | INFO     | nemo_curator.pipeline.pipeline:add_stage:61 - Added stage 'jsonl_reader' to pipeline 'ndd_data_generation'
[09:48:20] [WARNING] 🚨 You are trying to use a default model provider but your API keys are missing.
			Set the API key for the default providers you intend to use and re-initialize the Data Designer object.
			Alternatively, you can provide your own model providers during Data Designer object initialization.
			See https://nvidia-nemo.github.io/DataDesigner/concepts/models/model-providers/ for more information.


───────────────────────────────────────────────── Model Providers ─────────────────────────────────────────────────
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name        ┃ Endpoint                              ┃ API Key                                                   ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ nvidia      │ https://integrate.api.nvidia.com/v1   │ * 'NVIDIA_API_KEY' not set in environment variables *     │
│ openai      │ https://api.openai.com/v1             │ * 'OPENAI_API_KEY' not set in environment variables *     │
│ openrouter  │ https://openrouter.ai/api/v1          │ OPENROUTER_API_KEY                                        │
└─────────────┴───────────────────────────────────────┴───────────────────────────────────────────────────────────┘

2026-03-17 09:48:20.990 | INFO     | nemo_curator.pipeline.pipeline:add_stage:61 - Added stage 'DataDesignerStage' to pipeline 'ndd_data_generation'
2026-03-17 09:48:21.031 | INFO     | nemo_curator.pipeline.pipeline:add_stage:61 - Added stage 'jsonl_writer' to pipeline 'ndd_data_generation'


Pipeline: ndd_data_generation
Description: Generate synthetic text data using Nemo Data Designer
Stages: 3

Stage 1: jsonl_reader
  Resources: 1.0 CPUs
  Batch size: 1
  Outputs:
    Output attributes: data
    Output columns: diagnosis, patient_summary
Stage 2: DataDesignerStage
  Resources: 1.0 CPUs
  Batch size: 1
  Inputs:
    Required attributes: data
  Outputs:
    Output attributes: data
Stage 3: jsonl_writer
  Resources: 1.0 CPUs
  Batch size: 1
  Inputs:
    Required attributes: data
  Outputs:
    Output attributes: data



### Run the Pipeline

User can add `NVIDIA_API_KEY` by running `os.environ["NVIDIA_API_KEY"] = "YOUR_API_KEY"` if needed.

In [7]:
client = RayClient()  # change as needed
client.start()

2026-03-17 09:48:21.439 | WARNING  | nemo_curator.core.client:start:115 - No monitoring services are running. Please run the `start_prometheus_grafana.py` script from nemo_curator/metrics folder to setup monitoring services separately.
2026-03-17 09:48:21.460 | INFO     | nemo_curator.core.utils:init_cluster:187 - Ray start command: ray start --head --node-ip-address 127.0.1.1 --port 6381 --metrics-export-port 8083 --dashboard-host 127.0.0.1 --dashboard-port 8267 --ray-client-server-port 20001 --temp-dir /tmp/ray --disable-usage-stats --block
2026-03-17 09:48:21.461 | DEBUG    | nemo_curator.core.utils:check_ray_responsive:40 - Verifying Ray cluster is responsive, using RAY_ADDRESS=127.0.1.1:6381
2026-03-17 09:48:21.461 | DEBUG    | nemo_curator.core.utils:check_ray_responsive:47 - running 'ray status' command
/tmp/huvu-venvs/curator_env/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn'

2026-03-17 09:48:22,338	INFO usage_lib.py:447 -- Usage stats collection is disabled.
2026-03-17 09:48:22,338	INFO scripts.py:936 -- Local node IP: 127.0.1.1


2026-03-17 09:48:25.652 | DEBUG    | nemo_curator.core.utils:check_ray_responsive:72 - Ray cluster is not responsive ('ray status' command failed)
2026-03-17 09:48:26.153 | DEBUG    | nemo_curator.core.utils:check_ray_responsive:47 - running 'ray status' command
2026-03-17 09:48:27.225 | DEBUG    | nemo_curator.core.utils:check_ray_responsive:60 - Ray cluster is not responsive ('No cluster status' returned or Error in output)
2026-03-17 09:48:27.727 | DEBUG    | nemo_curator.core.utils:check_ray_responsive:47 - running 'ray status' command
2026-03-17 09:48:28.793 | DEBUG    | nemo_curator.core.utils:check_ray_responsive:60 - Ray cluster is not responsive ('No cluster status' returned or Error in output)
2026-03-17 09:48:29.295 | DEBUG    | nemo_curator.core.utils:check_ray_responsive:47 - running 'ray status' command
2026-03-17 09:48:30.369 | DEBUG    | nemo_curator.core.utils:check_ray_responsive:60 - Ray cluster is not responsive ('No cluster status' returned or Error in output)
2026

2026-03-17 09:48:35,252	SUCC scripts.py:975 -- --------------------
2026-03-17 09:48:35,252	SUCC scripts.py:976 -- Ray runtime started.
2026-03-17 09:48:35,252	SUCC scripts.py:977 -- --------------------
2026-03-17 09:48:35,252	INFO scripts.py:979 -- Next steps
2026-03-17 09:48:35,252	INFO scripts.py:982 -- To add another node to this Ray cluster, run
2026-03-17 09:48:35,252	INFO scripts.py:985 --   ray start --address='127.0.1.1:6381'
2026-03-17 09:48:35,252	INFO scripts.py:996 -- To connect to this Ray cluster:
2026-03-17 09:48:35,252	INFO scripts.py:998 -- import ray
2026-03-17 09:48:35,252	INFO scripts.py:999 -- ray.init(_node_ip_address='127.0.1.1')
2026-03-17 09:48:35,252	INFO scripts.py:1013 -- To submit a Ray job using the Ray Jobs CLI:
2026-03-17 09:48:35,252	INFO scripts.py:1014 --   RAY_API_SERVER_ADDRESS='http://127.0.0.1:8267' ray job submit --working-dir . -- python my_script.py
2026-03-17 09:48:35,252	INFO scripts.py:1023 -- See https://docs.ray.io/en/latest/cluster/runn

2026-03-17 09:48:35.712 | DEBUG    | nemo_curator.core.utils:check_ray_responsive:47 - running 'ray status' command
2026-03-17 09:48:36.781 | DEBUG    | nemo_curator.core.utils:check_ray_responsive:68 - Ray cluster IS responsive


In [ ]:
print("Starting synthetic data generation pipeline...")
start_time = time.time()
pipeline.run()
end_time = time.time()

elapsed_time = end_time - start_time
print("\nPipeline completed!")
print(f"Total execution time: {elapsed_time:.2f} seconds ({elapsed_time / 60:.2f} minutes)")

2026-03-17 09:48:37.289 | INFO     | nemo_curator.pipeline.pipeline:build:70 - Planning pipeline: ndd_data_generation
2026-03-17 09:48:37.290 | INFO     | nemo_curator.pipeline.pipeline:_decompose_stages:106 - Decomposing composite stage: jsonl_reader
2026-03-17 09:48:37.290 | INFO     | nemo_curator.pipeline.pipeline:_decompose_stages:120 - Expanded 'jsonl_reader' into 2 execution stages


Starting synthetic data generation pipeline...


2026-03-17 09:48:37.940 | INFO     | nemo_curator.backends.xenna.executor:execute:135 - Execution mode: STREAMING
2026-03-17 09:48:37,941	INFO worker.py:1669 -- Using address 127.0.1.1:6381 set in the environment variable RAY_ADDRESS
2026-03-17 09:48:37,967	INFO worker.py:1810 -- Connecting to existing Ray cluster at address: 127.0.1.1:6381...
2026-03-17 09:48:38,200	INFO worker.py:2004 -- Connected to Ray cluster. View the dashboard at 127.0.0.1:8267 
/tmp/huvu-venvs/curator_env/lib/python3.10/site-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
2026-03-17 09:48:39.640 | INFO     | nemo_curator.backends.xenna.adapter:required_resources:43 - Resources: Resources(cpus=0.5, gpu_memory_gb=0.0, nvdecs=0, nvencs=0, entire_gpu=False, gpus


Pipeline completed!
Total execution time: 211.23 seconds (3.52 minutes)


### Inspect the Output

We can find the output results stored in `./synthetic_output` directory.

In [15]:
output_files = get_all_file_paths_under(OUTPUT_PATH, recurse_subdirectories=True, keep_extensions=".jsonl")

print(f"Generated data saved to: {OUTPUT_PATH}")
for file_path in output_files:
    print(f"  - {Path(file_path).name}")

all_data_frames = [pd.read_json(f, lines=True) for f in output_files]

Generated data saved to: ./synthetic_output
  - 2a32c234df7d.jsonl
  - 383a455f5a8d.jsonl
  - 44767c13dcd9.jsonl
  - 572222cbb959.jsonl
  - 5ed1794c3d4a.jsonl
  - 642cda8b7e5b.jsonl
  - 6af70db0176e.jsonl
  - 6d23d6c30b3e.jsonl
  - 6eedcc8ded3c.jsonl
  - 7080f45e14c9.jsonl
  - 7b44d21ee99f.jsonl
  - 7d220206227d.jsonl
  - 7ec98cc83a2c.jsonl
  - 884f8629a7fd.jsonl
  - 8928c29fad4e.jsonl
  - c7fde8d4971e.jsonl
  - cc45e91592c4.jsonl
  - ead3a316b626.jsonl
  - f7f60d292b3e.jsonl
  - fb1bb122478d.jsonl


In [17]:
if all_data_frames:
    combined_df = pd.concat(all_data_frames, ignore_index=True)
    print(f"Total records generated: {len(combined_df)}")
    combined_df.head()

Total records generated: 200


Inspecting output .jsonl files content, we can find the generated text from Nvidia's LLM endpoint corresponding to the input prompt.

In [20]:
print("=" * 50)
print("Sample of generated documents:")
print("=" * 50)

for i, df in enumerate(all_data_frames[:3]):
    print(f"\nFile {i + 1}: {Path(output_files[i]).name}")
    print(f"Number of documents: {len(df)}")
    print("\nGenerated text (showing first record):")
    for j, row in enumerate(df.head(1).to_dict(orient="records")):
        print(f"Document {j + 1}:")
        for key, value in row.items():
            print(f"[{key}]:")
            print(f"{value}")
        print("-" * 40)

Sample of generated documents:

File 1: 2a32c234df7d.jsonl
Number of documents: 10

Generated text (showing first record):
Document 1:
[diagnosis]:
malaria
[patient_summary]:
I have a high fever, chills, and severe itching. I also have a headache, excessive sweating, nausea, and muscle aches.
[patient_sampler]:
{'uuid': 'd1a3836d-fd90-4a5e-8d19-cbb795067c59', 'locale': 'en_US', 'first_name': 'Angelica', 'last_name': 'Gates', 'middle_name': None, 'sex': 'Female', 'street_number': '125', 'street_name': 'Justin Brook', 'city': 'New Haleyhaven', 'state': 'Kansas', 'postcode': '76140', 'age': 20, 'birth_date': '2005-05-25', 'country': 'Ecuador', 'marital_status': 'divorced', 'education_level': 'some_college', 'unit': '', 'occupation': 'Freight forwarder', 'phone_number': '001-279-291-5066', 'bachelors_field': 'no_degree'}
[doctor_sampler]:
{'uuid': 'e47de38f-4332-412a-a30e-cf096289c597', 'locale': 'en_US', 'first_name': 'Richard', 'last_name': 'King', 'middle_name': None, 'sex': 'Male', 'st